In [1]:
!pip install google-generativeai



In [3]:
import os
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, clear_output

def build_system_prompt():
    return """
You are BBloom Assistant, a helpful and friendly chatbot.

BBloom is a cloud-based smart farming system designed mainly for blueberry growers,
but it can also support other plants and crops.

The goal of BBloom is to help users monitor plant health, detect diseases early,
improve irrigation decisions, reduce water waste and increase crop quality.

Main features of BBloom:

- IoT sensors that monitor:
  * Soil moisture
  * Air humidity
  * Temperature
  * Light intensity

- AI image analysis:
  * Detect plant diseases
  * Detect pests
  * Analyze plant health from photos
  * Provide confidence score for diagnosis

- Smart recommendations:
  * Irrigation recommendations
  * Fertilizing recommendations
  * Disease treatment recommendations
  * Personalized advice based on plant type and sensor data

- Dashboard:
  * Real-time sensor information
  * Plant health status
  * Alerts and notifications
  * Historical trends and graphs

- Garden/Farm Map:
  * Green = Healthy
  * Yellow = Warning
  * Red = Problem

- Alerts:
  * Low soil moisture
  * Extreme temperature
  * Sensor failure
  * Suspected disease or pest

- Vacation Mode:
  * Helps manage irrigation automatically while the user is away.

- Gamification:
  * Daily challenges
  * Weekly challenges
  * Points
  * Badges
  * Leaderboard

Typical users:
- Blueberry growers
- Home gardeners
- Greenhouse owners
- Farmers

If users ask questions about blueberries, plant care, irrigation,
diseases, sensors, AI analysis, alerts, cloud services or BBloom features,
provide helpful answers.

Answer clearly, briefly and professionally.
"""

class GeminiChatbot:
    def __init__(self, model_name="gemini-2.5-flash"):
        api_key = ""
        if not api_key:
            raise RuntimeError("Missing GEMINI_API_KEY (set it as an environment variable).")
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_name=model_name, system_instruction=build_system_prompt())
        self.chat = self.model.start_chat(history=[])

    def reply(self, text: str) -> str:
        r = self.chat.send_message(text)
        return (r.text or "").strip()

bot = GeminiChatbot()

# UI
title = widgets.HTML("<h3>Gemini Chatbot</h3>")
query_box = widgets.Text(
    placeholder="הקלד שאילתה כאן...",
    description="Query:",
    layout=widgets.Layout(width="80%")
)
send_btn = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear")
output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="10px", height="300px", overflow="auto"))

history = []

def render():
    with output:
        clear_output(wait=True)
        for who, msg in history:
            print(f"{who}: {msg}\n")

def on_send(_):
    text = query_box.value.strip()
    if not text:
        return
    history.append(("You", text))
    query_box.value = ""
    render()
    try:
        ans = bot.reply(text)
    except Exception as e:
        ans = f"Error calling Gemini: {e}"
    history.append(("Bot", ans))
    render()

def on_clear(_):
    history.clear()
    render()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

display(title, widgets.HBox([query_box, send_btn, clear_btn]), output)
render()


HTML(value='<h3>Gemini Chatbot</h3>')

Output(layout=Layout(border='1px solid #ddd', height='300px', overflow='auto', padding='10px'))